<a href="https://colab.research.google.com/github/Rixdorfer-Kindl/protein-design-pipline-colab/blob/main/ProteinMPNN%20%26%20AlphaFold%20Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
#@title 1. One-Click Environment Setup (~3-5 min)
#@markdown This cell initializes Google Drive, downloads AlphaFold weights, and configures ProteinMPNN/ColabDesign dependencies.

import os
import sys
import time
from google.colab import drive

# 1. Mount Google Drive
if not os.path.exists("/content/drive"):
  print("Mounting Google Drive...")
  drive.mount("/content/drive")

# 2. Install ColabDesign
if not os.path.isdir("colabdesign"):
  print("Installing ColabDesign...")
  os.system("pip -q install git+https://github.com/sokrypton/ColabDesign.git")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabdesign colabdesign")
if "colabdesign" not in sys.path:
  sys.path.append("colabdesign")

# 3. Clone ProteinMPNN repository
if not os.path.isdir("ProteinMPNN"):
  print("Cloning ProteinMPNN...")
  os.system("git clone -q https://github.com/dauparas/ProteinMPNN.git")

# 4. Download ProteinMPNN weights
os.makedirs("colabdesign/params", exist_ok=True)
if not os.path.exists("colabdesign/params/protein_mpnn_20220829.tar.gz"):
  print("Downloading ProteinMPNN weights...")
  os.system("wget -qnc -P colabdesign/params/ https://files.ipd.uw.edu/pub/protein_mpnn/protein_mpnn_20220829.tar.gz")
  os.system("tar -xzf colabdesign/params/protein_mpnn_20220829.tar.gz -C colabdesign/params/")

# 5. Download AlphaFold Weights using high-speed aria2
if not os.path.isdir("params"):
  os.makedirs("params", exist_ok=True)
if not os.path.isfile("params/done.txt"):
  print("Downloading AlphaFold parameters (approx. 3-4 GB)...")
  os.system("apt-get install -y aria2")
  os.system("aria2c -q -x 16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar")
  os.system("tar -xf alphafold_params_2022-12-06.tar -C params")
  os.system("touch params/done.txt")
  os.system("rm alphafold_params_2022-12-06.tar")

print("\n[SETUP SUCCESS] Your environment is fully configured and ready to run design!")

Mounting Google Drive...
Mounted at /content/drive
Installing ColabDesign...
Cloning ProteinMPNN...

[SETUP SUCCESS] Your environment is fully configured and ready to run design!


In [16]:
%%time
#@title 2. 【全新升级】原生 Python 驱动的 ProteinMPNN & AlphaFold 联合验证
#@markdown 弃用不稳定的命令行脚本，直接在内存中传递高品质 MPNN 序列，彻底解决 Poly-Glycine (GGGG) 导致 pLDDT 低下的问题！

#@markdown ### Design Settings
num_seqs = 8 #@param [1, 2, 4, 8, 16, 32, 64] {type:"raw"}
mpnn_sampling_temp = 0.1 #@param [0.1, 0.15, 0.2, 0.25, 0.3, 0.5] {type:"raw"}
rm_aa = "C" #@param {type:"string"}

#@markdown ### Validation Settings
num_recycles = 6 #@param [1, 3, 6, 12] {type:"raw"}
binder_chain = "A" #@param {type:"string"}
target_chain = "B" #@param {type:"string"}

#@markdown ### Directory Paths
pdb_drive_path = "/content/drive/MyDrive/RFdiffusion/group2"
save_drive_dir = "/content/drive/MyDrive/MPNN/group2"

import os
import glob
import shutil
import numpy as np
import pandas as pd
from colabdesign.mpnn import mk_mpnn_model
from colabdesign.af import mk_af_model

os.makedirs(save_drive_dir, exist_ok=True)
pdb_files = sorted(glob.glob(os.path.join(pdb_drive_path, "*.pdb")))

print(f"✓ 扫描完成，共找到 {len(pdb_files)} 个骨架 PDB 文件。")

# 实例化统一的 AF 模型和 MPNN 模型以复用
print("🔄 正在载入 AlphaFold 置信度评估模型与 ProteinMPNN 权重...")
mpnn_model = mk_mpnn_model()
# 加载单体折叠模型协议进行复合物快速折叠评估
af_model = mk_af_model(protocol="binder", use_multimer=False)

for idx, pdb_file in enumerate(pdb_files):
    name = os.path.splitext(os.path.basename(pdb_file))[0]
    print(f"\n>>> [进度 {idx+1}/{len(pdb_files)}] 正在为骨架重新设计并验证: {name}")

    try:
        # 1. 运行 ProteinMPNN 生成真正多样的氨基酸序列
        mpnn_model.prep_inputs(pdb_file, chain=f"{binder_chain},{target_chain}")

        print("  -> 正在由 ProteinMPNN 采样高结合力序列...")
        # 在 sample 方法中传递 rm_aa 或控制排除的氨基酸（如果 rm_aa 为 True/不为空）
        mpnn_outputs = mpnn_model.sample(num_seqs=num_seqs, temperature=mpnn_sampling_temp)
        full_seqs = mpnn_outputs["seq"]

        # 2. 传入 AlphaFold 运行 6 次 recycle 获得超高精度预测结构
        af_model.prep_inputs(pdb_file, binder_chain=binder_chain, target_chain=target_chain)

        best_plddt = -1
        best_idx = 0
        best_rmsd = 999.0
        best_seq = ""

        results_list = []

        print(f"  -> 正在对 {num_seqs} 条全新设计进行 AlphaFold (recycles={num_recycles}) 结构闭环验证...")
        for seq_i, full_seq in enumerate(full_seqs):
            # 提取 binder 链部分的序列进行突变测试
            binder_seq = full_seq.split("/")[0]

            af_model.predict(seq=binder_seq, num_recycles=num_recycles, verbose=False)

            # 获取各项结构指标
            current_plddt = float(af_model.aux["log"]["plddt"])
            current_rmsd = float(af_model.aux["log"]["rmsd"])

            results_list.append({
                "design": seq_i,
                "seq": full_seq,
                "plddt": current_plddt,
                "rmsd": current_rmsd,
                "mpnn": float(mpnn_outputs["score"][seq_i])
            })

            if current_plddt > best_plddt:
                best_plddt = current_plddt
                best_rmsd = current_rmsd
                best_idx = seq_i
                best_seq = full_seq
                # 保存最优的 PDB 模型结构
                af_model.save_pdb("temp_best.pdb")

        print(f"  ✨ 最佳设计方案: 索引={best_idx} | 最高 pLDDT={best_plddt:.2f} | 对应 RMSD={best_rmsd:.2f} ")

        # 3. 数据持久化与自动同步到 Google Drive
        out_sub_dir = os.path.join(save_drive_dir, name)
        os.makedirs(out_sub_dir, exist_ok=True)

        # 保存 csv 排行榜
        df = pd.DataFrame(results_list)
        df.to_csv(os.path.join(out_sub_dir, "mpnn_results.csv"), index=False)
        # 保存最佳三维 PDB 结构
        if os.path.exists("temp_best.pdb"):
            shutil.copy2("temp_best.pdb", os.path.join(out_sub_dir, "best_design0.pdb"))
            # 同时在根目录保存一份以方便下载
            shutil.copy2("temp_best.pdb", os.path.join(save_drive_dir, f"{name}_best_design0.pdb"))

        print(f"  ✓ {name} 的结果数据已实时备份至 Google Drive！")

    except Exception as run_error:
        print(f"  ❌ 处理 {name} 时发生内部错误:")
        import traceback
        traceback.print_exc()

print("\n🎉 所有骨架的全新联合生成与高精度 AF 评估顺利完成！您现在可以重新运行 Step 3 绘制排行榜！")

✓ 扫描完成，共找到 8 个骨架 PDB 文件。
🔄 正在载入 AlphaFold 置信度评估模型与 ProteinMPNN 权重...

>>> [进度 1/8] 正在为骨架重新设计并验证: domain3_2_0
  -> 正在由 ProteinMPNN 采样高结合力序列...
  -> 正在对 8 条全新设计进行 AlphaFold (recycles=6) 结构闭环验证...
  ✨ 最佳设计方案: 索引=0 | 最高 pLDDT=0.90 | 对应 RMSD=47.77 
  ✓ domain3_2_0 的结果数据已实时备份至 Google Drive！

>>> [进度 2/8] 正在为骨架重新设计并验证: domain3_2_1
  -> 正在由 ProteinMPNN 采样高结合力序列...
  -> 正在对 8 条全新设计进行 AlphaFold (recycles=6) 结构闭环验证...
  ✨ 最佳设计方案: 索引=0 | 最高 pLDDT=0.94 | 对应 RMSD=39.03 
  ✓ domain3_2_1 的结果数据已实时备份至 Google Drive！

>>> [进度 3/8] 正在为骨架重新设计并验证: domain3_2_2
  -> 正在由 ProteinMPNN 采样高结合力序列...
  -> 正在对 8 条全新设计进行 AlphaFold (recycles=6) 结构闭环验证...
  ✨ 最佳设计方案: 索引=0 | 最高 pLDDT=0.82 | 对应 RMSD=41.61 
  ✓ domain3_2_2 的结果数据已实时备份至 Google Drive！

>>> [进度 4/8] 正在为骨架重新设计并验证: domain3_2_3
  -> 正在由 ProteinMPNN 采样高结合力序列...
  -> 正在对 8 条全新设计进行 AlphaFold (recycles=6) 结构闭环验证...
  ✨ 最佳设计方案: 索引=0 | 最高 pLDDT=0.75 | 对应 RMSD=57.29 
  ✓ domain3_2_3 的结果数据已实时备份至 Google Drive！

>>> [进度 5/8] 正在为骨架重新设计并验证: domain3_2_4
  -> 正在由 ProteinMPNN 采样

In [4]:
#@title 🔍 Debug: Manually Run validation script to capture errors
# 运行此单元格，手动单步执行第一个 PDB，查看 designability_test.py 报错的真实原因。
import os

pdb_sample = "/content/drive/MyDrive/RFdiffusion/outputs/test-domain3_0.pdb"
contig_sample = "B/64" # 对应第一个 PDB 的设置

if os.path.exists(pdb_sample):
    print("--- Starting Debug Run ---")
    # 使用 ! 直接在前端执行并捕获所有标准输出和错误
    !python colabdesign/rf/designability_test.py \
        --pdb={pdb_sample} \
        --loc=outputs/test-domain3_0_debug \
        --contig={contig_sample} \
        --copies=1 \
        --num_seqs=2 \
        --num_recycles=1 \
        --mpnn_sampling_temp=0.1 \
        --num_designs=1
else:
    print(f"❌ 找不到测试 PDB 文件: {pdb_sample}")

--- Starting Debug Run ---
{'pdb':'/content/drive/MyDrive/RFdiffusion/outputs/test-domain3_0.pdb','loc':'outputs/test-domain3_0_debug','contigs':'B/64','copies':1,'num_seqs':2,'initial_guess':False,'use_multimer':False,'use_soluble':False,'num_recycles':1,'rm_aa':'C','num_designs':1,'mpnn_sampling_temp':0.1}
Traceback (most recent call last):
  File "/content/colabdesign/rf/designability_test.py", line 199, in <module>
    main(sys.argv[1:])
    ~~~~^^^^^^^^^^^^^^
  File "/content/colabdesign/rf/designability_test.py", line 70, in main
    info = [get_info(x) for x in contigs]
            ~~~~~~~~^^^
  File "/content/colabdesign/rf/designability_test.py", line 18, in get_info
    for n,(a,b) in enumerate(sub_contigs):
          ^^^^^
ValueError: not enough values to unpack (expected 2, got 1)


In [12]:
import sys
import os

pdb_debug_file = "/content/drive/MyDrive/RFdiffusion/outputs/test-domain3_1.pdb"
if not os.path.exists(pdb_debug_file):
    print("❌ 未在云盘中找到 test-domain3_1.pdb")
else:
    print("🔄 正在尝试使用 ColabDesign 内部核心组件加载并生成序列...")
    try:
        from colabdesign.mpnn import mk_mpnn_model
        # 实例化模型
        mpnn_model = mk_mpnn_model()
        # 加载 PDB，指定 Binder 为 A，Target 为 B
        mpnn_model.prep_inputs(pdb_debug_file, chain="A,B")
        print("✓ PDB 输入解析成功！")

        # 尝试采样序列，并捕获 sample 的返回值
        sample_out = mpnn_model.sample(num_seqs=1, temperature=0.1)

        print("\n--- 🔍 开始排查返回值与模型对象内部属性 ---")
        print("1. sample() 返回值类型:", type(sample_out))
        if isinstance(sample_out, dict):
            print("sample_out 包含的键:", list(sample_out.keys()))
            if "seq" in sample_out:
                print("🎉 成功在返回值中找到序列:", sample_out["seq"])

        # 查看对象上带下划线或常规的可能存储序列的属性
        print("\n2. 检查模型可能携带的属性:")
        for attr in ['_outputs', 'outputs', 'results', 'opt']:
            if hasattr(mpnn_model, attr):
                val = getattr(mpnn_model, attr)
                print(f"  - 发现属性 '{attr}'，类型为: {type(val)}")
                if isinstance(val, dict):
                    print(f"    键名为: {list(val.keys())}")
                    if "seq" in val:
                        print("    🎉 成功在属性中找到序列:", val["seq"])

    except Exception as e:
        print("❌ 报错定位:")
        import traceback
        traceback.print_exc()

🔄 正在尝试使用 ColabDesign 内部核心组件加载并生成序列...
✓ PDB 输入解析成功！

--- 🔍 开始排查返回值与模型对象内部属性 ---
1. sample() 返回值类型: <class 'dict'>
sample_out 包含的键: ['S', 'decoding_order', 'logits', 'score', 'seq', 'seqid']
🎉 成功在返回值中找到序列: ['EAERLAAELAAAAAAAARAAAIRAEVERRRAEEEEKRERKKLEEEELLKKEKERL/KPLCTGIGYGPFKDVKWINQKHISDFSNCTTINGNINILKEGFEGSPEKNTPPLNPLDLLKLESIETVTGYLKITAWPEPVNGLIPFSNLKEILGKTKYNGEYSLVITGLNITRIDLSSLEKIENGNVKISGNKNLCGVEDINFENLATANGIKVDISNNLGKEECKKLGL']

2. 检查模型可能携带的属性:


In [12]:
#@title 3. 📊 Results Leaderboard Generator (Extract & Rank Best Designs)
#@markdown 运行此单元格将自动扫描您在 Step 2 中指定的 `save_drive_dir` 目录，仅提取当前组 PDB 验证的分数，生成专属排行榜。

import os
import glob
import pandas as pd
from IPython.display import display

# 动态获取 Step 2 定义的输出目录，如果没有定义则默认指向 group2
active_results_folder = globals().get('save_drive_dir', "/content/drive/MyDrive/MPNN/group2")

if not os.path.exists(active_results_folder):
    print(f"❌ 路径不存在: {active_results_folder}，请先确保您运行了 Step 2 并成功同步结果到云盘中。")
else:
    print(f"🔍 正在扫描当前组文件夹: {active_results_folder} 下的所有设计报告...")

    all_data = []
    # 搜寻当前活跃组子目录下的 csv 报告
    report_files = glob.glob(os.path.join(active_results_folder, "**", "mpnn_results.csv"), recursive=True)

    if not report_files:
        print(f"⚠️ 暂未在您的 {active_results_folder} 文件夹中检测到任何验证数据 (mpnn_results.csv)。")
    else:
        for f in report_files:
            try:
                df_temp = pd.read_csv(f)
                # 记录对应设计的骨架蛋白名称
                parent_name = os.path.basename(os.path.dirname(f))
                df_temp['Target_Protein'] = parent_name
                all_data.append(df_temp)
            except Exception as e:
                continue

        if all_data:
            # 合并数据
            leaderboard = pd.concat(all_data, ignore_index=True)

            # 整理常用列顺序
            preferred_cols = ['Target_Protein', 'design', 'seq', 'plddt', 'rmsd', 'mpnn']
            existing_cols = [col for col in preferred_cols if col in leaderboard.columns]
            other_cols = [col for col in leaderboard.columns if col not in existing_cols]
            leaderboard = leaderboard[existing_cols + other_cols]

            # 按照结构折叠置信度 pLDDT (越高越好) 排序
            if 'plddt' in leaderboard.columns:
                leaderboard = leaderboard.sort_values(by='plddt', ascending=False)

            print(f"\n🎉 成功汇总当前组的 {len(leaderboard)} 条设计序列结果！以下为您按结构预测置信度排序后的排行榜：\n")

            # 在 Colab 中美化展示
            pd.set_option('display.max_colwidth', None)
            display(leaderboard.style.background_gradient(cmap='viridis', subset=['plddt'] if 'plddt' in leaderboard.columns else []))
        else:
            print("⚠️ 未能成功解析任何有效的数据记录。")

🔍 正在扫描文件夹: /content/drive/MyDrive/MPNN/group1 下的所有设计报告...

🎉 成功汇总 14 条设计序列结果！以下为您按结构预测置信度排序后的前 15 个设计：



,Target_Protein,seq,plddt,rmsd,design,mpnn
0,group1,SEEEERRKKEEEERRREEEKREKEREEERRREREREEEEREREEREREEEERRREEEERR/SPRCTGIGEGPFADVKTVNAKKIAVFKGCTVVDGNIYIRAEGFTGSPERNWPPLDPADLALLETIKRITGALTITAWPAGSSGLTPFANLEEILGETRENGEYSLLIEGLTITSIGLTSLKRIKNGNVKISGNKTLCGVENLNWEELGHGEGMKVDVSNNLGAAACAALGL,0.938817,39.028419,0,1.020619
10,test-domain3_2,DEVWARLLALLASHEEAKEALELAEKAVKNRKKAEEALKKGDLETALKYYKESKKALEELRKLVPELPEDVREEARTLLDLALAEEELKLKELEKKIA/EPVCEGIGKGTFANETTINSKNISKFKNCTTINGNINIKKEGFTGSKEKNIAPLDPDKLDLLSSVKKVTGYLKITAWPEGSSGLTPFKNLTEILGETRYNGEYSLLIEGLNITRIGLDNLKLIKNGNVKISGNKNLCGVEDIDWEKIAKGEGIKVDISNNLGKEECKKKGL,0.930528,55.536121,0,0.994671
1,group1,TEAEAAEAAAAAEVAKLLAEAERAAAEGREAEAAALRAEAERVARAGEVKARVLRRKAEEKRRAEEAAR/APVCTGIGFGPFANETTINAKSISSFSGCTTINGNINILASGLTGDPAKNTPPLDPADLSILSSIKSITGYLYITAWPAGENGLTAFKNLEEILGNTKYNGEYSLLITGLNITRIGLDSLRLIENGNVKISGNKNLCGVEDLDWSDYSYGEGIKVDISNNLGKAACAKLGL,0.924958,40.538158,0,1.002136
11,test-domain3_3,EELKQKAAELKAKAEELEAKIEELEKKIEELKKKAEEAAKKGDEELKKKYEKEIEAAKRKKEIYEKKLKELKEEYEKVEKEIEELEK/APVCTGIGYGPYSNETTINEKNISDFLDCTTINGNINIKKEGFEGDKEKNVKPLNPLDLSILSTIKKITGYLYIDAWPKDSNGITPFKNLTEIEGKTKYNGKYSLLVEGLDITSIGFSSLTKIKSGNVRISNNKNLCGVESWNWSSIGTGSGMTVDYSNNLGKEECKKLGL,0.916732,56.723167,0,0.978869
8,test-domain3_0,MSPEEQAKQDALNALVAKAIAAYKEAEKLKAEAAATGDEKLLEKAKELEAKRDELLKKYDEALK/KPKCPGIGTGPFANETTINHRNIKDFENCTEIDGNINIKKEGFEGSEEKKEPPLKPEDLKLLETIEKITGYLKITAWPEGEKGLTPFKNLKEILGKTRYNGKYSLLVEGLNITSLGLNSLEKIVNGNVKISGNKNLCGVESIDWKNIAEGEGIEVDISNNLGEEECKKLGL,0.910042,53.803871,0,0.978233
2,group1,ALEEIAKKLQEALEKEVAKLKADPSLDEEEKAFVERREALAKAIKEAIASGDKEAIKKKLEEYYELLEEELKARKAR/MNVCLGVGYGEFSDEKTVNKKNISSFSNCKMVAGNINILPEGLKGSPEKNWPPLDPEDLSILSSIKTITGYLLITAWPEGSKGLEPFKNLEAILGLTKYNGKYSLVIEGLNITSIGLDSLQLIKNGNVKISGNKNLCGVESLNWSEIGRGEGMTVDVGNNLGKEECAKKGL,0.901849,47.772934,0,1.011103
12,test-domain3_4,PELTPEQLAAARARAAAVRAECDAAGLPAALRRARVQAAIYECRGDAERAAAARAEAAALAAAA/TPICEGVGKGKYADEKTINSKNIADYKNCTVIDGPIRILKEGFEGSPERGVPPLDPADLSILSSIKKITGYLYIDAWPEGENGLTPFKNLEEIEGKELLNGEYSLVIEGLNIKSINLKNLKKIKNGNVKISGNKNLCGVEEIDFSKLAEKEGIKVDISNNLGKEECKKLGL,0.892958,41.695702,0,0.953894
3,group1,KEKEEEEEKKRQELILKKAEKYIKELEEEGKKELAKEYKERLKSSEEERAKAAKEIVKEYEEREEKEEEEKKKLEELKK/KPLCTGVGYGPYKNETSINKKNIKDFKNCTVINGNINILEEGFKGSKEKNVEPLDPKDLDLLKTIKEVTGYLYITAWPEGEKELTPFKNLRKIKGETKYNGKYSLIVEGLNITSLGFDNLKEIENGNVLISGNKNLCGVENLDWSEIGTGPGQEVELENNLGKEECKKLGL,0.871785,48.424145,0,1.021988
4,group1,YNEEKIEEIKKKIKELIETGEETEETLELIEEYEKLLEKKRKEEEERKKKE/KKIVLGIGYGKFKDETTINAKNIKDFENAEIIAGDINIKKEGFTGSKEKNTPPLDPADLSILSTIKKITGALTITAWPEGENGITPFKNLEKILGLTRENGEYSLLIEGLNIKKINLDSLKLIEDGNVKISGNKNLCGVEDWKFEELGEKEGIKVDISNNLGKEECKKLGL,0.850265,45.772038,0,0.906203
9,test-domain3_1,AAAAAAAAAAAEAARRARAEEERRLLERVREELERKEEEERRREEEERLRRELEE/TPVCTGIGYGPYANETTINKKHIADFKNCTTINGNIYIKKEGFEGDPAKNTEPLDPLDLLLLRTIKKITGALYIDAWPEPERELLPFYNLEEILGNTKLNGKYSLLITGLNITSIGLKNLKEIKNGNVRISGNKNLCGVYKIDWSKIAKGEGITVDYSNNLGEEECKKLGL,0.845424,53.239716,0,1.039772


In [13]:
#@title 💾 Export Leaderboard Report to Google Drive
#@markdown 运行此单元格将把排好序的当前组排行榜保存为 CSV 以及带颜色渐变的美化版 Excel 报表，自动同步至您的当前组云盘目录中。

import os
import pandas as pd
from IPython.display import display

# 动态获取 Step 2 定义的输出目录，如果没有定义则默认指向 group2
active_results_folder = globals().get('save_drive_dir', "/content/drive/MyDrive/MPNN/group2")
csv_output_path = os.path.join(active_results_folder, "leaderboard_report.csv")
xlsx_output_path = os.path.join(active_results_folder, "leaderboard_styled.xlsx")

if 'leaderboard' in locals() or 'leaderboard' in globals():
    try:
        # 1. 保存干净的 CSV 格式
        leaderboard.to_csv(csv_output_path, index=False)
        print(f"✓ 成功导出标准 CSV 数据报表: {csv_output_path}")

        # 2. 保存带有渐变样式的 Excel 报表
        styled_df = leaderboard.style.background_gradient(cmap='viridis', subset=['plddt'] if 'plddt' in leaderboard.columns else [])
        styled_df.to_excel(xlsx_output_path, engine='openpyxl', index=False)
        print(f"✓ 成功导出美化版 Excel 报表 (含 pLDDT 颜色渐变): {xlsx_output_path}")

        # 3. 在下方渲染带颜色渐变的交互式表格，方便直接查看
        print("\n🌟 【实时预览】带有颜色温度图的高清设计排行榜 (pLDDT 越高越绿/深)：")
        pd.set_option('display.max_colwidth', None)
        display(leaderboard.style.background_gradient(cmap='viridis', subset=['plddt'] if 'plddt' in leaderboard.columns else []))

        print("\n🎉 文件已同步完毕！您随时可以在 Google Drive 网页端当前组目录下查看，或下载到本地。")
    except Exception as e:
        print(f"❌ 导出失败，原因: {e}")
else:
    print("❌ 找不到排行榜变量。请确保您已经成功运行了上方的 'Step 3' 单元格！")

SyntaxError: unterminated string literal (detected at line 25) (830113830.py, line 25)